In [2]:
import pandas as pd
import numpy as np

In [ ]:
# LOAD NLP DATASET
df = pd.read_csv(
    "../data/nlp_features.csv"
)

print(df.shape)

(4661, 45)


In [ ]:
# WEBSITE AVAILABLE
#No website = suspicious

df["website_available"] = (

    df["company_website"]

    .fillna("")

    .str.len()

    .gt(5)

    .astype(int)

)

In [ ]:
# =====================================================
# EMAIL AVAILABLE (from raw field)
# =====================================================
# Real recruiters usually provide contact. Using raw recruiter_email field only.

df["email_available"] = (
    df["recruiter_email"]
    .fillna("")
    .str.len()
    .gt(5)
    .astype(int)
)

In [ ]:
# =====================================================
# SALARY AVAILABLE (from raw fields)
# =====================================================
# No salary info = lower transparency. Checking both salary_clean and stipend_clean.

df["salary_available"] = (
    (
        df["salary_clean"]
        .fillna("")
        .astype(str)
        .str.len()
        > 0
    )
    |
    (
        df["stipend_clean"]
        .fillna("")
        .astype(str)
        .str.len()
        > 0
    )
).astype(int)

In [ ]:
# =====================================================
# DESCRIPTION LENGTH (raw field metric)
# =====================================================

df["description_length"] = (
    df["description"]
    .fillna("")
    .str.len()
)

# =====================================================
# SHORT DESCRIPTION FLAG
# =====================================================
# Tiny descriptions are suspicious.

df["short_description_flag"] = (
    df["description_length"] < 30
).astype(int)

In [ ]:
# =====================================================
# SKILLS MISSING
# =====================================================
# No skills listed = poor quality job. Using has_skills flag from NB2.

df["skills_missing"] = (
    df["has_skills"] == 0
).astype(int)

In [ ]:
# =====================================================
# COMPANY ABOUT MISSING
# =====================================================
# No company information = risk

df["company_about_missing"] = (
    df["company_about"]
    .fillna("")
    .str.len()
    .lt(20)
).astype(int)

In [ ]:
# =====================================================
# DEADLINE PRESSURE SCORE
# =====================================================
# "Apply immediately!", "Only few days left!" are common scam tactics.
# Score: 2 if posted ≤2 days ago, 1 if 2-7 days ago, 0 otherwise.

df["deadline_pressure_score"] = 0

df.loc[
    df["posted_days_ago"] <= 2,
    "deadline_pressure_score"
] = 2

df.loc[
    (df["posted_days_ago"] > 2) & (df["posted_days_ago"] <= 7),
    "deadline_pressure_score"
] = 1

In [ ]:
# CONTACT RISK

df["contact_risk"] = (

    df["contact_score"]

    * 2

)

In [ ]:
# =====================================================
# TRUST SCORE (from raw/available fields)
# =====================================================
# Higher = more trustworthy. Sum of 5 availability indicators.

df["trust_score"] = (
    df["website_available"]
    + df["email_available"]
    + df["salary_available"]
    + df["has_description"]
    + df["has_skills"]
)

In [ ]:
# FRAUD SCORE V2

df["fraud_score_v2"] = (

    df["risk_score"]

    +

    df["missing_info_score"] * 5

    +

    df["short_description_flag"] * 10

    +

    df["company_about_missing"] * 10

    +

    df["contact_risk"] * 5

    +

    df["deadline_pressure_score"] * 5

)

In [ ]:
# FRAUD LABEL

df["fraud_label"] = np.where(

    df["fraud_score_v2"] >= 50,

    "SCAM",

    "GENUINE"

)

In [17]:
# FRAUD COUNTS

df["fraud_label"].value_counts()

fraud_label
GENUINE    4240
SCAM        421
Name: count, dtype: int64

In [ ]:
# TOP RISK JOBS

df[

    [

        "title",

        "company",

        "fraud_score_v2",

        "fraud_label"

    ]

].sort_values(

    by="fraud_score_v2",

    ascending=False

).head(20)

,title,company,fraud_score_v2,fraud_label
2698,community management - internship wfh & part time,nayepankh foundation,90,SCAM
2805,campus ambassador - internship wfh & part time,base,85,SCAM
1446,telecalling - internship,search digitally,85,SCAM
4270,customer support,hucon solutions india pvt.ltd.,80,SCAM
4443,accountant,tharwani infrastructure,80,SCAM
3930,sr customer service executive,vibrantzz management services hiring for inter...,80,SCAM
3208,telecaller bpo call center hsc fresher custome...,altruist technologies,70,SCAM
3999,urgent hiring us non it recruiter,sharda consultancy services,70,SCAM
3991,urgent hiring ota manager travel industry,sharda consultancy services,70,SCAM
4288,international customer support voice process m...,tech mahindra ltd.,70,SCAM


In [19]:
# SAVE FINAL FEATURES

df.to_csv(

    "../data/final_features.csv",

    index=False

)

print(

    "Saved: final_features.csv"

)

Saved: final_features.csv
